# Building Image Generation Applications (Azure OpenAI)

There's more to LLMs than text generation. You can also generate images from text descriptions. Images as a modality are useful across MedTech, architecture, tourism, game development, marketing, and more. In this lesson we work with today's **GPT Image** models on Microsoft Foundry.

## Learning goals

By the end of this lesson you'll be able to:

- Explain what image generation is and where it's useful.
- Understand the `gpt-image` model family and how it differs from the legacy DALL·E models.
- Build an image generation application and edit images.

## What is image generation?

Image generation models create images from a text prompt. Modern models such as `gpt-image` learn the relationship between text and images during training, then iteratively turn random noise into an image that matches your prompt.

Two well-known families of image models are:

- **`gpt-image` (OpenAI)** — the current generation used in this lesson. It supports text-to-image generation and image editing (inpainting with a mask).
- **Midjourney** — a popular third-party model with its own service and Discord-based workflow.

> The older OpenAI image models — **DALL·E 2** and **DALL·E 3** — are legacy. DALL·E 3 is no longer available for new deployments, and features like `create_variation` only existed in DALL·E 2. Use the `gpt-image` models for new applications.

On Microsoft Foundry, **`gpt-image-2`** is the latest and most capable image model and is the recommended default. `gpt-image-1.5` and `gpt-image-1-mini` are also generally available.

> **Important:** `gpt-image` models return the generated image as **base64** (`b64_json`), not as a URL. Your code decodes the base64 string to bytes and saves it — there's no image URL to download.


## Building your first image generation application

So what does it take to build an image generation application? You need the following libraries:

- **python-dotenv**, you're highly recommended to use this library to keep your secrets in a *.env* file away from the code.
- **openai**, this library is what you will use to interact with the OpenAI API.
- **pillow**, to work with images in Python.

If not done already, follow the instructions on the [Microsoft Learn](https://learn.microsoft.com/azure/ai-foundry/openai/how-to/create-resource?pivots=web-portal&WT.mc_id=academic-105485-koreyst) page to create a Microsoft Foundry resource and model. Select **gpt-image-2** as the model (the latest Azure OpenAI image model; DALL·E is legacy).

1. Create a file *.env* with the following content:

    ```text
    AZURE_OPENAI_ENDPOINT=<your endpoint>
    AZURE_OPENAI_API_KEY=<your key>
    AZURE_OPENAI_DEPLOYMENT="gpt-image-2"
    ```

    Locate this information in the Microsoft Foundry portal for your resource in the "Deployments" section.



1. Collect the above libraries in a file called *requirements.txt* like so:

    ```text
    python-dotenv
    openai
    pillow
    ```


1. Next, create virtual environment and install the libraries:


In [ ]:
# create virtual env
! python3 -m venv venv
# activate environment
! source venv/bin/activate
# install libraries
# pip install -r requirements.txt, if using a requirements.txt file 
! pip install python-dotenv openai pillow

> [!NOTE]
> For Windows, use the following commands to create and activate your virtual environment:

    ```bash
    python3 -m venv venv
    venv\Scripts\activate.bat
    ````

1. Add the following code in file called *app.py*:

    ```python
    import os
    import base64
    from PIL import Image
    import dotenv
    from openai import AzureOpenAI, BadRequestError

    # import dotenv
    dotenv.load_dotenv()

    # configure Azure OpenAI (Microsoft Foundry) client.
    # Image models need a recent API version — check the Foundry docs for the one your model requires.
    client = AzureOpenAI(
      azure_endpoint = os.environ["AZURE_OPENAI_ENDPOINT"],
      api_key=os.environ['AZURE_OPENAI_API_KEY'],
      api_version = "2025-04-01-preview"
      )

    try:
        # Create an image by using the image generation API
        generation_response = client.images.generate(
            prompt='Bunny on horse, holding a lollipop, on a foggy meadow where it grows daffodils',    # Enter your prompt text here
            size='1024x1024',
            n=1,
            model=os.environ['AZURE_OPENAI_DEPLOYMENT']   # e.g. "gpt-image-2"
        )
        # Set the directory for the stored image
        image_dir = os.path.join(os.curdir, 'images')

        # If the directory doesn't exist, create it
        if not os.path.isdir(image_dir):
            os.mkdir(image_dir)

        # Initialize the image path (note the filetype should be png)
        image_path = os.path.join(image_dir, 'generated-image.png')

        # gpt-image models return the image as base64 (b64_json), not a URL
        image_b64 = generation_response.data[0].b64_json
        generated_image = base64.b64decode(image_b64)
        with open(image_path, "wb") as image_file:
            image_file.write(generated_image)

        # Display the image in the default image viewer
        image = Image.open(image_path)
        image.show()

    # catch exceptions
    except BadRequestError as err:
        print(err)

    ```

Let's explain this code:

- First, we import the libraries we need, including the OpenAI library, the dotenv library, the base64 module, and the Pillow library.

    ```python
    import os
    import base64
    from PIL import Image
    import dotenv
    from openai import AzureOpenAI, BadRequestError
    ```

- Next, we load the environment variables from the *.env* file.

    ```python
    # import dotenv
    dotenv.load_dotenv()
    ```

- After that, we configure the Azure OpenAI (Microsoft Foundry) client.

    ```python
    client = AzureOpenAI(
      azure_endpoint = os.environ["AZURE_OPENAI_ENDPOINT"],
      api_key=os.environ['AZURE_OPENAI_API_KEY'],
      api_version = "2025-04-01-preview"
      )
    ```

- Next, we generate the image:

    ```python
    generation_response = client.images.generate(
        prompt='Bunny on horse, holding a lollipop, on a foggy meadow where it grows daffodils',    # Enter your prompt text here
        size='1024x1024',
        n=1,
        model=os.environ['AZURE_OPENAI_DEPLOYMENT']
    )
    ```

    `gpt-image` models return the image as a **base64** string in `data[0].b64_json`. We decode it to bytes and write it to a file — there's no URL to download.

    ```python
    image_b64 = generation_response.data[0].b64_json
    generated_image = base64.b64decode(image_b64)
    with open(image_path, "wb") as image_file:
        image_file.write(generated_image)
    ```

- Lastly, we open the image and use the standard image viewer to display it:

    ```python
    image = Image.open(image_path)
    image.show()
    ```

### More details on generating the image

Let's look at the parameters of `images.generate`:

- **prompt**, is the text prompt used to generate the image. Here it's "Bunny on horse, holding a lollipop, on a foggy meadow where it grows daffodils".
- **size**, is the size of the generated image (for example `1024x1024`, `1536x1024`, `1024x1536`, or `"auto"`).
- **n**, is the number of images generated. Here we generate one.
- **model**, is your image model deployment name (for example `gpt-image-2`).

> Image models don't take a `temperature` parameter — that's a text-generation control. To get variety, call the API again; to reduce variety, make your prompt more specific.

## Additional capabilities of image generation

You've seen how to generate an image with a few lines of Python. `gpt-image` models can also **edit** an existing image. By providing an image, an optional **mask** (which marks the area to change), and a prompt, you can alter part of an image — for example, add a hat to our bunny.

```python
response = client.images.edit(
  model=os.environ['AZURE_OPENAI_DEPLOYMENT'],
  image=open("base_image.png", "rb"),
  mask=open("mask.png", "rb"),
  prompt="An image of a rabbit with a hat on its head.",
  n=1,
  size="1024x1024"
)
# edits are also returned as base64
edited_image = base64.b64decode(response.data[0].b64_json)
```

The base image contains just the rabbit; the final image has the hat on the rabbit.


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**Disclaimer**:
This document has been translated using AI translation service [Co-op Translator](https://github.com/Azure/co-op-translator). While we strive for accuracy, please be aware that automated translations may contain errors or inaccuracies. The original document in its native language should be considered the authoritative source. For critical information, professional human translation is recommended. We are not liable for any misunderstandings or misinterpretations arising from the use of this translation.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
